# Imports

In [1]:
"""Run elasticity on the layered/faulted grid with hole."""
import numpy as np
from pathlib import Path

from create_grid import create_layered_grid_with_hole
from common_functions import solve_and_export

#folder = Path(__file__).parent
folder = Path.cwd()



In [2]:


"""Run the layered-grid-with-hole elasticity workflow.

Parameters
----------
None
    The script uses fixed settings defined inside the function.

Returns
-------
None
    Solution files are written to the results directory.
"""
mesh_size = 0.025

# Step 1: generate layered/faulted grid with hole and layer id.
# Use function defaults for geometry, fault, and hole settings.
sd, cell_layer = create_layered_grid_with_hole(mesh_size=mesh_size, center=(1.5, 0.25))



c:\Users\steto\anaconda3.1\envs\poresct4amm2526\Lib\site-packages\porepy\grids\mdg_generation.py:924: UserWarning: Found 51 fractures outside the domain boundary
  warn(f"Found {num_missing} fractures outside the domain boundary")


In [3]:


# Step 2: solve elasticity and export displacement + layer id.
folder_export = folder / "results"
u = solve_and_export(
    sd,
    folder_export,
    export_name="sol_layered_hole",
    cell_data=[("layer", cell_layer.astype(float))],
)



# measurement

We plot the displacement in the point P(0,1,0). You should write a routine to extract it when the mesh change.

In [4]:
print(u[0][60])
print(u[1][60])



4.635762482906495e-05
-9.931874611270088e-06


# Signature of the function

In [5]:
def GroupSolutionCP1_prova( measured_displacement ):

    # measured_displacement is a Tuple(float,float), e.g. (0.003, 0.00045)

    # your implementation

    # format of the output
    center_location = (0.5,0.5)

    return center_location

In [5]:
from scipy.optimize import minimize
import os
import sys
from contextlib import redirect_stdout

In [7]:
def extract_displacement(sd, u, target_point=(0.0, 1.0)):
    """
    Find the node geometrically closer to target_point and extract its changing components.
    ----------------------------------------------------------------------------------------------------
    Trova il nodo geometricamente più vicino al target_point ed estrae le sue componenti di spostamento.
    """
    # Compute square distance between all nodes and (0,1)
    dist_sq = (sd.nodes[0, :] - target_point[0])**2 + (sd.nodes[1, :] - target_point[1])**2
    
    # Taking the index of the node with least distance
    idx = np.argmin(dist_sq)
    
    # Givin' [u_x, u_y] for that node
    return np.array([u[0, idx], u[1, idx]])


def evaluation_cost_function(center, mesh_size, measured_disp, export_folder):
    cx, cy = center
    
    # Penalty for boundary (security)
    if cx < 0.25 or cx > 1.75 or cy < 0.25 or cy > 0.75:
        return 1e6

    try:
        with open(os.devnull, 'w') as f, redirect_stdout(f):
            sd, _ = create_layered_grid_with_hole(mesh_size=mesh_size, center=(cx, cy))
            u = solve_and_export(sd, export_folder, export_name="temp_inv", cell_data=None)
            simulated_disp = extract_displacement(sd, u, target_point=(0.0, 1.0))
        
        # Computing error
        error = np.linalg.norm(simulated_disp - np.array(measured_disp))
        return error
        
    except Exception:
        return 1e6

In [8]:
def GroupSolutionCP1(measured_displacement):
    """
    Solve inverse problem with a Multi-Grid + Multi-Start approach.
    Risolve il problema inverso con approccio Multi-Grid con Multi-Start.
    """
    print("\nStarting inverse Multi-Grid with Multi-Start...")
    export_folder = Path.cwd() / "opt_temp_results"
    export_folder.mkdir(exist_ok=True)
    
    coarse_mesh_size = 0.1
    refined_mesh_size = 0.04
    
    # =====================================================================
    # PHASE 1: global research 
    # =====================================================================
    print("--> PHASE 1: Scanning Coarse Grid (6x5)...")
    x_vals = np.linspace(0.3, 1.7, 6)
    y_vals = np.linspace(0.3, 0.7, 5)
    
    # Saving all coarse grid results
    coarse_results = []
    
    for cx in x_vals:
        for cy in y_vals:
            err = evaluation_cost_function(
                center=(cx, cy), 
                mesh_size=coarse_mesh_size, 
                measured_disp=measured_displacement, 
                export_folder=export_folder
            )
            coarse_results.append({
                'center': (cx, cy),
                'error': err
            })

    # Sorting results from the best (least error) to the worse        
    coarse_results.sort(key=lambda x: x['error'])
    
    # MULTI-START: taking the best 2 results
    top_candidates = [res['center'] for res in coarse_results[:2]]
    
    print(f"    Identified the two best starting-points:")
    for i, cand in enumerate(top_candidates):
        print(f"      {i+1}) {cand} (Error: {coarse_results[i]['error']:.4e})")

    # =====================================================================
    # PHASE 2: Local Multi-Optimization with Refined Grid
    # =====================================================================
    print("\n--> PHASE 2: Refined Multi-Start Optimization...")
    
    best_overall_center = None
    best_overall_error = float('inf')
    
    # Nelder-Mead from each of the 2 best candidates
    for idx, start_center in enumerate(top_candidates):
        print(f"    Nelder-Mead execution {idx+1}/2 starting from {start_center}...")
        
        res = minimize(
            fun=evaluation_cost_function,
            x0=start_center,
            args=(refined_mesh_size, measured_displacement, export_folder),
            method='Nelder-Mead',
            options={
                'xatol': 0.005,
                'fatol': 1e-8,
                'maxiter': 15  
            }
        )
        
        # Saving if the optimization have find a better result than the previous
        if res.fun < best_overall_error:
            best_overall_error = res.fun
            best_overall_center = tuple(res.x)
            
        print(f"      Ended with an error: {res.fun:.4e}")

    print(f"\n✅ Optimization ended! Final least error: {best_overall_error:.4e}")
    print(f"\nFinal coordinates estimated: {best_overall_center}")
    
    return best_overall_center

### Test su più casi / Multi-Case Test

In [9]:
def run_multiple_inversion_tests(num_tests=5):
    print(f"=========================================================")
    print(f"🚀🌍 STARTING EVALUATION ROBUSTNESS ON {num_tests} RANDOM CAVITIES")
    print(f"=========================================================\n")
    

    # ========================================================
    # CREATION OF GROUND TRUTH
    # ========================================================

    # Folder to not overscript or misread the files
    export_folder = Path.cwd() / "test_multiple_results"
    export_folder.mkdir(exist_ok=True)
    
    target_point = (0.0, 1.0)
    true_mesh_size = 0.025
    
    # List the keep results and compute final statistics
    results_summary = []
    
    for i in range(num_tests):
        print(f"\n" + "-"*50)
        print(f" TEST {i+1} OF {num_tests}")
        print("-" * 50)
        
        # 1. Create random coordinates
        true_cx = np.random.uniform(0.3, 1.7)
        true_cy = np.random.uniform(0.3, 0.7)
        true_center = (true_cx, true_cy)
        
        print(f"1. Creation Ground Truth: Cavity hidden in ({true_cx:.4f}, {true_cy:.4f})...")
        
        try:
            # Creation real physical mesh
            sd_true, cell_layer_true = create_layered_grid_with_hole(
                mesh_size=true_mesh_size, 
                center=true_center
            )
            
            # FEM solver
            u_true = solve_and_export(
                sd_true, 
                export_folder, 
                export_name=f"sol_ground_truth_test_{i+1}",
                cell_data=[("layer", cell_layer_true.astype(float))]
            )
            
            # Extraction measured displacement
            measured_displacement = extract_displacement(sd_true, u_true, target_point=target_point)
            
        except Exception as e:
            print(f"⚠️ Critical mesher error for the coordinates {true_center}. Going on with the next test.\nError: {e}")
            continue

        # ========================================================
        # CALL GroupSolutionCP1
        # ========================================================
        
        print("2. Starting inversion...")
        # 2. Givin the measured_displacement to the inverse algorithm
        estimated_center = GroupSolutionCP1(tuple(measured_displacement))
        
        # 3. Computing the error (euclidean distance)
        error_x = abs(true_center[0] - estimated_center[0])
        error_y = abs(true_center[1] - estimated_center[1])
        dist_totale = np.sqrt(error_x**2 + error_y**2)
        
        print(f"\n TEST RESULTS {i+1}:")
        print(f"   REAL center:   ({true_cx:.4f}, {true_cy:.4f})")
        print(f"   ESTIMATE center: ({estimated_center[0]:.4f}, {estimated_center[1]:.4f})")
        print(f"   Spatial error: {dist_totale:.4f} m")
        
        # Saving report for final table
        results_summary.append({
            "test_id": i+1,
            "true": true_center,
            "est": estimated_center,
            "error": dist_totale
        })

    # =====================================================================
    # FINAL REPORT
    # =====================================================================
    print("\n" + "="*50)
    print("3. FINAL EVALUATION REPORT")
    print("="*50)
    
    if len(results_summary) == 0:
        print("No test ended succesfully")
        return
        
    for res in results_summary:
        # Use 0.05m (5 cm) as acceptability threshold
        status = "✅ PASSED" if res['error'] < 0.05 else "⚠️ IMPRECISE"
        
        t_x, t_y = res['true']
        e_x, e_y = res['est']
        
        print(f"Test {res['test_id']}: Real ({t_x:.3f}, {t_y:.3f}) | Estimate ({e_x:.3f}, {e_y:.3f}) | Error: {res['error']:>6.4f}m | {status}")
    
    avg_error = np.mean([res['error'] for res in results_summary])
    max_error = np.max([res['error'] for res in results_summary])
    
    print("-" * 50)
    print(f"Mean Errore:   {avg_error:.4f} m")
    print(f"Maximum Errore: {max_error:.4f} m")
    print("=" * 50)

# Start of the cycle
if __name__ == "__main__":
    run_multiple_inversion_tests(num_tests=5)

🚀🌍 STARTING EVALUATION ROBUSTNESS ON 5 RANDOM CAVITIES


--------------------------------------------------
 TEST 1 OF 5
--------------------------------------------------
1. Creation Ground Truth: Cavity hidden in (0.4449, 0.5942)...

2. Starting inversion...

Starting inverse Multi-Grid with Multi-Start...
--> PHASE 1: Scanning Coarse Grid (6x5)...
    Identified the two best starting-points:
      1) (np.float64(0.58), np.float64(0.39999999999999997)) (Error: 3.8586e-05)
      2) (np.float64(0.58), np.float64(0.3)) (Error: 4.8098e-05)

--> PHASE 2: Refined Multi-Start Optimization...
    Nelder-Mead execution 1/2 starting from (np.float64(0.58), np.float64(0.39999999999999997))...
      Ended with an error: 2.2170e-06
    Nelder-Mead execution 2/2 starting from (np.float64(0.58), np.float64(0.3))...
      Ended with an error: 1.8499e-05

✅ Optimization ended! Final least error: 2.2170e-06

Final coordinates estimated: (np.float64(0.4435668945312504), np.float64(0.5993603515625016))


* quando il centro è a sinistra della faglia, l'algoritmo l'azzecca al millimetro!
* quando il centro è a destra della faglia, l'errore cresce al centimetro
* tempo: ~3min/caso